In [1]:
import logging
import shutil
import socket
import sys
from datetime import UTC, datetime
from math import prod
from pathlib import Path
from pprint import pformat
from time import perf_counter
from bidict import bidict
import torch
import pyiqa


device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

import numpy as np
from memory_tempfile import MemoryTempfile as MemoryTemper
from tifffile import TiffFile

from comptest import imagemagick, libtiff, tifffile
from comptest.constants import (
    DATASETS_PROJECTIONS_DIR,
    DATASETS_RECONSTRUCTIONS_DIR,
)
from comptest.types import (
    DTYPE_ANY,
    DTYPE_BOOL,
    DTYPE_FLOAT,
    DTYPE_UINT8,
    DTYPE_UINT8_UINT16,
    DTYPE_UINT32,
    CompressionTest,
    FileStackCompressionConfig,
    StructuralSimilarityIndexMetrics,
)

import tifffile as tifffile_lib
from pprint import pprint
from comptest.image_transform import DataTransform

%load_ext autoreload
%autoreload 2

# Parameters

In [2]:
reverse_configs: bool = False
target_configs: str | None = None
is_papermill: bool = False
run_id: str = "none"

# Setup

In [3]:
def configure_logging(log_dir: Path) -> tuple[logging.Logger, Path]:
    log_file = log_dir / "compression.log"

    configured_logger = logging.getLogger(__name__)
    configured_logger.setLevel(logging.DEBUG)
    configured_logger.propagate = False

    # Avoid duplicate handlers when notebook cells are rerun.
    for handler in list(configured_logger.handlers):
        if getattr(handler, "_compression_notebook_handler", False):
            configured_logger.removeHandler(handler)
            handler.close()

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    stdout_handler = logging.StreamHandler(sys.stdout)
    stdout_handler.setLevel(logging.DEBUG)
    stdout_handler.setFormatter(formatter)
    stdout_handler._compression_notebook_handler = True

    file_handler = logging.FileHandler(log_file, mode="a", encoding="utf-8")
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)
    file_handler._compression_notebook_handler = True

    configured_logger.addHandler(stdout_handler)
    configured_logger.addHandler(file_handler)

    return configured_logger, log_file

In [4]:
PROJECT_DIR = Path().resolve()
OUTPUT_DIR = PROJECT_DIR / "output"
assert OUTPUT_DIR.is_dir()

mem_temper = MemoryTemper(filesystem_types=["tmpfs"])

In [5]:
DATASET = "Canislupus_CZM_712"
# DATASET_DIR = DATASETS_DIR / DATASET
DATASET_DIR = DATASETS_RECONSTRUCTIONS_DIR / "CIL_basic_binned_2x2" / DATASET
assert DATASET_DIR.is_dir()

SCRATCH_INPUT_FILES = sorted(
    [
        *DATASET_DIR.glob("*.tif"),
        *DATASET_DIR.glob("*.tiff"),
    ]
)

# pprint(SCRATCH_INPUT_FILES)

In [6]:
# Load input files into memory
# !!! Note: We currently don't (always) clean this up after use!
temp_dir_path = (
    Path(mem_temper.gettempdir()) / "s250250-compression-tests-filestack-in-memory"
)
print(temp_dir_path)

if temp_dir_path.exists():
    shutil.rmtree(temp_dir_path)

temp_dir_path.mkdir(parents=True, exist_ok=True)
INPUT_FILES = []
for file in SCRATCH_INPUT_FILES:
    shutil.copy(file, temp_dir_path / file.name)
    INPUT_FILES.append(temp_dir_path / file.name)


/dev/shm/s250250-compression-tests-filestack-in-memory


In [7]:
# Check dtype is the same for all
INPUT_DTYPE = None
_input_shapes = []
for file in INPUT_FILES:
    with TiffFile(file) as tif:
        dtype = tif.pages[0].dtype
        _input_shapes.append(tif.pages[0].shape)
        if INPUT_DTYPE is None:
            INPUT_DTYPE = dtype
        else:
            assert dtype == INPUT_DTYPE, (
                f"Input files have different dtypes: {dtype} != {INPUT_DTYPE}"
            )

INPUT_DTYPE

dtype('float32')

In [8]:
for shape in _input_shapes:
    assert shape == _input_shapes[0], (
        f"Input files have different shapes: {shape} != {_input_shapes[0]}"
    )

INPUT_SHAPE = (len(_input_shapes), *_input_shapes[0])
INPUT_SHAPE

(1000, 1000, 1000)

In [9]:
# Load up data into a single array so we can get some information easily with numpy
data = np.stack([tifffile_lib.imread(file) for file in INPUT_FILES], axis=0)
data_min = data.min()
data_max = data.max()
data_mean = data.mean()
data_std = data.std()
data_range = data_max - data_min

# NOTE: Disabled for now since I don't like how this is very sensitive to outliers
data_transform: DataTransform | None = None  # Needed for SSIM
target_data_range = (0.0, 1.0)
# if INPUT_DTYPE in {np.float16, np.float32, np.float64}:
#     data_transform = DataTransform.infer(data, target_data_range)

del data

# Run

In [10]:
host = socket.gethostname()
TEST_CATEGORY = "full"
TEST_HOST = host
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

TEST_OUTPUT_DIR = (
    OUTPUT_DIR
    / "file-stack"
    / f"{TEST_CATEGORY}_{RUN_TIMESTAMP}_{TEST_HOST}_{run_id}"
    / DATASET
)
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logger, LOG_FILE = configure_logging(TEST_OUTPUT_DIR)

logger.info("Initialized compression test run")
logger.info("Host: %s", host)
logger.info("Dataset: %s", DATASET)
logger.info("Test output directory: %s", TEST_OUTPUT_DIR)
logger.info("Log file: %s", LOG_FILE)

2026-08-11 12:41:04 | INFO     | Initialized compression test run
2026-08-11 12:41:04 | INFO     | Host: n-62-21-102
2026-08-11 12:41:04 | INFO     | Dataset: Canislupus_CZM_712
2026-08-11 12:41:04 | INFO     | Test output directory: /zhome/1a/a/219376/Code/qim-naturarv3d-compression/output/file-stack/full_2026-08-11_12-41-04_n-62-21-102_none/Canislupus_CZM_712
2026-08-11 12:41:04 | INFO     | Log file: /zhome/1a/a/219376/Code/qim-naturarv3d-compression/output/file-stack/full_2026-08-11_12-41-04_n-62-21-102_none/Canislupus_CZM_712/compression.log


NameError: name 'CONFIGS' is not defined

In [ ]:
from comptest.configurations import (
    CONFIGS,
    CONFIGS_LOSSLESS,
    TIFFFILE_JPEGXL_LOSSY_CONFIGS_EFFORT_7_BIG,
)

BASE_EXTRA = {
    # "bulk_runner_kwargs": {"n_jobs": 16, "joblib_kwargs": {"prefer": "processes"}},
    "bulk_runner_kwargs": {"n_jobs": None, "joblib_kwargs": {"prefer": "processes"}},
}

if target_configs is not None:
    TARGET_CONFIGS = globals()[target_configs]
else:
    # TARGET_CONFIGS = CONFIGS
    # TARGET_CONFIGS = CONFIGS_LOSSLESS
    TARGET_CONFIGS = TIFFFILE_JPEGXL_LOSSY_CONFIGS_EFFORT_7_BIG

_reverser = -1 if reverse_configs else 1
TARGET_CONFIGS = TARGET_CONFIGS[::_reverser]

initial_size = sum(f.stat().st_size for f in INPUT_FILES)
tests = []

logger.info(
    "Number of configs: %d, lossy: %d, lossless: %d, reversed: %s",
    len(TARGET_CONFIGS),
    len([c for c in TARGET_CONFIGS if not c.lossless]),
    len([c for c in TARGET_CONFIGS if c.lossless]),
    reverse_configs,
)
logger.info("Input files: %d", len(INPUT_FILES))
logger.info("Initial input size: %d bytes", initial_size)


for config in TARGET_CONFIGS:
    try:
        if INPUT_DTYPE not in config.allowed_dtypes:
            logger.warning(
                "Skipping config %s because dtype %s is unsupported",
                config.name,
                INPUT_DTYPE,
            )
            continue

        logger.info("Starting config: %s", config.name)

        start_wall_dt = datetime.now(UTC)
        total_wall_time_start = perf_counter()
        extra = BASE_EXTRA.copy()

        # Put in tmpfs to avoid disk I/O bottleneck
        # This is automatically deleted at end of scope/context manager
        with mem_temper.TemporaryDirectory() as compress_temp_dir:
            logger.debug(
                "Created temporary directory for compression: %s",
                compress_temp_dir,
            )
            compress_temp_dir_path = Path(compress_temp_dir)

            # Run compression
            compress_start = perf_counter()
            compressed_files = [
                compress_temp_dir_path / (input_file.name) for input_file in INPUT_FILES
            ]
            logger.info("Compressing with config %s", config.name)
            compression_results = config.compressor[0](
                INPUT_FILES,
                compressed_files,
                *config.compressor[1],
                **config.compressor[2],
                **extra["bulk_runner_kwargs"],
            )
            compress_wall_time_seconds = perf_counter() - compress_start
            logger.info(
                "Compression completed for %s in %.3f seconds",
                config.name,
                compress_wall_time_seconds,
            )
            compress_user_time_seconds = sum(r.user_time for r in compression_results)
            compress_system_time_seconds = sum(
                r.system_time for r in compression_results
            )

            # Run decompression
            with mem_temper.TemporaryDirectory() as decompress_temp_dir:
                logger.debug(
                    "Created temporary directory for decompression: %s",
                    decompress_temp_dir,
                )
                decompress_temp_dir_path = Path(decompress_temp_dir)
                decompressed_files = [
                    decompress_temp_dir_path / (input_file.name)
                    for input_file in INPUT_FILES
                ]
                decompress_start = perf_counter()
                logger.info("Decompressing output from config %s", config.name)
                decompression_results = config.decompressor[0](
                    compressed_files,
                    decompressed_files,
                    *config.decompressor[1],
                    **config.decompressor[2],
                    **extra["bulk_runner_kwargs"],
                )
                decompress_wall_time_seconds = perf_counter() - decompress_start
                logger.info(
                    "Decompression completed for %s in %.3f seconds",
                    config.name,
                    decompress_wall_time_seconds,
                )
                decompress_user_time_seconds = sum(
                    r.user_time for r in decompression_results
                )
                decompress_system_time_seconds = sum(
                    r.system_time for r in decompression_results
                )

                # Calculate size before releasing from memory
                uncompressed_size = initial_size

                # Get individual file sizes
                images_input_sizes = {}
                images_uncompressed_sizes = {}
                images_compressed_sizes = {}
                for input_file, compressed_file, decompressed_file in zip(
                    INPUT_FILES, compressed_files, decompressed_files
                ):
                    images_input_sizes[input_file.name] = input_file.stat().st_size
                    images_uncompressed_sizes[input_file.name] = (
                        decompressed_file.stat().st_size
                    )
                    images_compressed_sizes[input_file.name] = (
                        compressed_file.stat().st_size
                    )

                assert sum(images_input_sizes.values()) == initial_size
                actual_uncompressed_size = sum(images_uncompressed_sizes.values())
                compressed_size = sum(images_compressed_sizes.values())

                # Run comparison
                logger.info("Comparing reconstructed files for config %s", config.name)
                comparisons = config.comparer[0](
                    INPUT_FILES,
                    decompressed_files,
                    *config.comparer[1],
                    **config.comparer[2],
                    **extra["bulk_runner_kwargs"],
                )

                was_actually_lossless = all(comparisons)
                logger.info(
                    "Comparison completed for %s: lossless=%s",
                    config.name,
                    was_actually_lossless,
                )

                # SSIM
                if was_actually_lossless:
                    ssim = None
                else:
                    logger.info("Calculating SSIM for config %s", config.name)
                    ssim_full = np.empty(INPUT_SHAPE, dtype=INPUT_DTYPE)
                    for i, (input_file, decompressed_file) in enumerate(
                        zip(INPUT_FILES, decompressed_files)
                    ):
                        with TiffFile(input_file) as tif:
                            input_image = tif.asarray()
                        with TiffFile(decompressed_file) as tif:
                            decompressed_image = tif.asarray()

                        # Note: disabled for now since this distorts data if outliers are present
                        if data_transform is not None:
                            input_image = data_transform.apply(input_image)
                            decompressed_image = data_transform.apply(
                                decompressed_image
                            )

                        ssim_full[i] = StructuralSimilarityIndexMetrics.get_ssim(
                            input_image, decompressed_image, data_range=data_range
                        )

                    ssim = StructuralSimilarityIndexMetrics.from_ndarray(ssim_full)
                    logger.info("SSIM calculated for config %s", config.name)
                    print(ssim_full.shape)

        # Get pixel count for bitrate
        n_elements = 0
        for input_file in INPUT_FILES:
            # Note that this does not actually read image data, so not super slow
            with TiffFile(input_file) as tif:
                for page in tif.pages:
                    n_elements += prod(page.shape)

        # Calculate quantities
        compression_wall_rate = uncompressed_size / compress_wall_time_seconds
        compression_user_rate = uncompressed_size / compress_user_time_seconds
        decompression_wall_rate = uncompressed_size / decompress_wall_time_seconds
        decompression_user_rate = uncompressed_size / decompress_user_time_seconds

        bitrate_uncompressed = 8 * initial_size / n_elements
        bitrate_compressed = 8 * compressed_size / n_elements

        compression_ratio = uncompressed_size / compressed_size
        space_saving = 1 - 1 / compression_ratio

        end_wall_dt = datetime.now(UTC)
        total_wall_time_seconds = perf_counter() - total_wall_time_start

        test = CompressionTest(
            **config.model_dump(),
            id=config.get_pathlike_str(),
            dataset_name=DATASET,
            dataset_path=DATASET_DIR,
            start_wall_dt=start_wall_dt,
            end_wall_dt=end_wall_dt,
            total_wall_time_seconds=total_wall_time_seconds,
            config=config,
            baseline_test=None,  # TODO: For now?
            host=host,
            compress_wall_time_seconds=compress_wall_time_seconds,
            compress_user_time_seconds=compress_user_time_seconds,
            compress_system_time_seconds=compress_system_time_seconds,
            decompress_wall_time_seconds=decompress_wall_time_seconds,
            decompress_user_time_seconds=decompress_user_time_seconds,
            decompress_system_time_seconds=decompress_system_time_seconds,
            uncompressed_size=uncompressed_size,
            actual_uncompressed_size=actual_uncompressed_size,
            compressed_size=compressed_size,
            n_elements=n_elements,
            compression_wall_rate=compression_wall_rate,
            compression_user_rate=compression_user_rate,
            decompression_wall_rate=decompression_wall_rate,
            decompression_user_rate=decompression_user_rate,
            bitrate_uncompressed=bitrate_uncompressed,
            bitrate_compressed=bitrate_compressed,
            compression_ratio=compression_ratio,
            space_saving=space_saving,
            was_actually_lossless=was_actually_lossless,
            ssim=ssim,
            images_input_sizes=images_input_sizes,
            images_uncompressed_sizes=images_uncompressed_sizes,
            images_compressed_sizes=images_compressed_sizes,
            runner_extra=extra,
        )

        tests.append(test)

        logger.debug("Test result:\n%s", pformat(test.summary(), sort_dicts=False))

        output_file = TEST_OUTPUT_DIR / f"test_{test.name}.json"
        with open(output_file, "w") as f:
            f.write(test.model_dump_json(indent=4, fallback=repr))

        logger.info(
            "Completed config %s: ratio=%.4f, space_saving=%.2f%%, result=%s",
            config.name,
            compression_ratio,
            space_saving * 100,
            output_file,
        )

    except Exception as e:
        if isinstance(e, KeyboardInterrupt):
            logger.warning("KeyboardInterrupt received, stopping tests")
            raise

        logger.exception("Error running test for config %s", config.name)

2026-08-07 13:54:20 | INFO     | Number of configs: 77, lossy: 77, lossless: 0, reversed: False
2026-08-07 13:54:20 | INFO     | Input files: 1000
2026-08-07 13:54:20 | INFO     | Initial input size: 4000134000 bytes
2026-08-07 13:54:20 | INFO     | Starting config: tifffile_jpegxl_d0.01_e1
2026-08-07 13:54:20 | DEBUG    | Created temporary directory for compression: /dev/shm/tmpcfpukn8y
2026-08-07 13:54:20 | INFO     | Compressing with config tifffile_jpegxl_d0.01_e1
2026-08-07 13:54:53 | INFO     | Compression completed for tifffile_jpegxl_d0.01_e1 in 33.487 seconds
2026-08-07 13:54:53 | DEBUG    | Created temporary directory for decompression: /dev/shm/tmpclgr7p3i
2026-08-07 13:54:53 | INFO     | Decompressing output from config tifffile_jpegxl_d0.01_e1
2026-08-07 13:55:13 | INFO     | Decompression completed for tifffile_jpegxl_d0.01_e1 in 19.548 seconds
2026-08-07 13:55:13 | INFO     | Comparing reconstructed files for config tifffile_jpegxl_d0.01_e1
2026-08-07 13:55:19 | INFO    

In [ ]:
# TIFFFILE_JPEGXL_CONFIGS[0]
# import tifffile

# example_input = INPUT_FILES[0]

# data = tifffile.imread(example_input)

# offset = -data.min()
# offset_data = data + offset
# scale = 1 / offset_data.max()
# scaled_data = offset_data * scale

# tifffile.imwrite(
#     "/tmp/test.tif", scaled_data, compression="jpegxl", compressionargs={"effort": 1}
# )


In [ ]:
# Clean up memory
CLEAR_MEMORY = True

if is_papermill or CLEAR_MEMORY:
    shutil.rmtree(temp_dir_path)